#### patch-h readout
Set the reading along English `h` at L12, token `t_inst`, to the English harmful level, then read the refusal signal along English `r` at L18 (last template token). Forward pass only, no generation.
- **patch-h**: does a loud, English-sized harm signal raise `r` for each language?
- **positive control**: English harmless + patch-h should raise `r` (otherwise a single-token patch is too weak).
- **random control**: same push size along a random direction should not.

In [ ]:
import json
from pathlib import Path

import torch
from harm_route.collect import collect_resid_batched
from harm_route.interventions import make_pos_mean_ablate, t_inst_index
from harm_route.models import load_bridge, load_tokenizer, model_config, pad_token_id
from harm_route.prompts import find_positions

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name = "qwen3-1.7b"
langs = ["eng_Latn", "npi_Deva", "srp_Cyrl", "srp_Latn"]
sets = ["harmful_val", "harmless_val"]
H_LAYER, R_LAYER = 12, 18

tok = load_tokenizer(name); kw = model_config(name)["template_kwargs"]; pad_id = pad_token_id(tok)
bridge = load_bridge(name, device="cpu")
A = {lang: torch.load(ROOT / f"data/cache/e01/{name}/{lang}_acts.pt") for lang in langs}   # no-patch activations
dirs = torch.load(ROOT / f"data/cache/e00/{name}_directions.pt")
h_hat = dirs["h"][H_LAYER] / dirs["h"][H_LAYER].norm()          # [d]
r_hat = dirs["r"][-1, R_LAYER] / dirs["r"][-1, R_LAYER].norm()  # [d]
prompts = {lang: json.load(open(ROOT / f"data/raw/e01/{name}/splits/{lang}.json")) for lang in langs}

Targets and scale, from English TRAIN (never val)

In [ ]:
E = A["eng_Latn"]
mu = (E["harmful_train"]["inst"][:, H_LAYER] @ h_hat).mean().item()    # English harmful level on h, L12 t_inst
r_lo = (E["harmless_train"]["last"][:, R_LAYER] @ r_hat).mean()         # English harmless level on r, L18
r_hi = (E["harmful_train"]["last"][:, R_LAYER] @ r_hat).mean()          # English harmful level on r

def scale(x):
    return (x - r_lo) / (r_hi - r_lo)                                   # 0 = English harmless, 1 = English harmful

g = torch.Generator().manual_seed(0)
u_hat = torch.randn(h_hat.shape[0], generator=g)
u_hat = u_hat / u_hat.norm()                                           # [d], random control direction
print(f"mu {mu:.2f} | r_lo {r_lo:.2f} r_hi {r_hi:.2f} | cos(u, h) {(u_hat @ h_hat).item():.3f}")

Loop: baseline (cache) vs patch-h vs random, every language and set

In [ ]:
def patch_hooks(v, level):
    """hooks for collect_resid_batched: set the reading along v at t_inst, L12, to level (per batch positions)"""
    return lambda ps_, shift: [(f"blocks.{H_LAYER}.hook_resid_post",
                                make_pos_mean_ablate(v, level, t_inst_index(ps_, shift)))]

res = {}   # res[(lang, st)] -> {"base", "patch", "rand"}: [N] scaled r at L18
for lang in langs:
    for st in sets:
        ps = [find_positions(tok, p, kw) for p in prompts[lang][st]]
        x_inst = A[lang][st]["inst"][:, H_LAYER]                         # [N, d], no patch
        base = scale(A[lang][st]["last"][:, R_LAYER] @ r_hat)          # [N]

        inst, post = collect_resid_batched(bridge, ps, pad_id, hooks=patch_hooks(h_hat, mu))
        assert torch.allclose(inst[:, H_LAYER] @ h_hat, torch.full((len(ps),), mu), atol=1e-2)   # patch landed
        assert torch.equal(inst[:, :H_LAYER], A[lang][st]["inst"][:, :H_LAYER])                  # earlier layers untouched
        patch = scale(post[:, -1, R_LAYER] @ r_hat)                    # [N]

        delta = mu - (x_inst @ h_hat).mean().item()                    # average push patch-h applied to this set
        mu_u = (x_inst @ u_hat).mean().item() + delta                  # same distance along the random direction
        _, post_u = collect_resid_batched(bridge, ps, pad_id, hooks=patch_hooks(u_hat, mu_u))
        rand = scale(post_u[:, -1, R_LAYER] @ r_hat)                   # [N]

        res[(lang, st)] = {"base": base, "patch": patch, "rand": rand, "delta": delta}

In [ ]:
print(f"{'lang':9s} {'set':13s} {'push':>6s} {'base':>6s} {'patch-h':>8s} {'random':>7s}   (r at L18, English scale: 0 = Eng harmless, 1 = Eng harmful)")
for (lang, st), r in res.items():
    print(f"{lang:9s} {st:13s} {r['delta']:6.1f} {r['base'].mean():6.2f} {r['patch'].mean():8.2f} {r['rand'].mean():7.2f}")

Per prompt: baseline vs patch-h (above the diagonal = r went up)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)
for ax, st in zip(axes, sets):
    for i, lang in enumerate(langs):
        r = res[(lang, st)]
        ax.scatter(r["base"], r["patch"], s=14, color=f"C{i}", label=lang)
    ax.axline((0, 0), slope=1, color="k", lw=0.8)
    ax.axhline(1, color="gray", lw=0.5, ls="--"); ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.set_title(f"{st}: r at L18 (English scale)"); ax.set_xlabel("baseline"); ax.set_ylabel("after patch-h")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

pushing the X language towards the english h did not do much work as we see in the plot above so we add that h information to all the instruction positions not only the t_inst

In [ ]:
def make_span_add(v, delta, start, end):
    """Hook: add delta * v_hat to tokens start[b]..end[b] (inclusive) in each row b; other tokens untouched."""
    v_hat = v / v.norm()
    def fn(resid, hook):
        if resid.shape[1] == 1:
            return resid
        pos = torch.arange(resid.shape[1], device=resid.device)
        s = start.to(resid.device)[:, None]
        e = end.to(resid.device)[:, None]
        mask = (pos >= s) & (pos <= e)
        push = delta * v_hat.to(resid.device, resid.dtype)
        return resid + mask[..., None].to(resid.dtype) * push
    return fn

def span_hooks(v, delta):
    """For collect_resid_batched: build the hook per batch, since padding shifts positions differently each batch."""
    def build(ps_, shift):
        start = torch.tensor([p["inst_start"] for p in ps_]) + shift
        end = torch.tensor([p["t_inst"] for p in ps_]) + shift
        return [(f"blocks.{H_LAYER}.hook_resid_post", make_span_add(v, delta, start, end))]
    return build

In [ ]:
E = A["eng_Latn"]
gap = ((E["harmful_train"]["inst"][:, H_LAYER] @ h_hat).mean()
        - (E["harmless_train"]["inst"][:, H_LAYER] @ h_hat).mean()).item()
gap

In [ ]:
lang, st = "eng_Latn", "harmless_val"
ps = [find_positions(tok, p, kw) for p in prompts[lang][st]]
x_inst = A[lang][st]["inst"][:, H_LAYER]                         # [N, d], no-push activations at t_inst
base = scale(A[lang][st]["last"][:, R_LAYER] @ r_hat)            # [N], r with no push (from cache)
n_tok = torch.tensor([p["t_inst"] - p["inst_start"] + 1 for p in ps]).float()
print(f"Δ(1x) = {gap:.1f} | instruction tokens per prompt: mean {n_tok.mean():.1f}, min {n_tok.min():.0f}")

gate = {}
for mult in [1, 2, 4]:
    d = mult * gap
    inst, post = collect_resid_batched(bridge, ps, pad_id, hooks=span_hooks(h_hat, d))
    assert torch.allclose(inst[:, :H_LAYER], A[lang][st]["inst"][:, :H_LAYER], atol=1e-3)  # earlier layers untouched
    assert torch.allclose(inst[:, H_LAYER] @ h_hat, x_inst @ h_hat + d, atol=1e-2)        # push landed: reading +d
    _, post_u = collect_resid_batched(bridge, ps, pad_id, hooks=span_hooks(u_hat, d))     # same size, random direction
    gate[mult] = {"h": scale(post[:, -1, R_LAYER] @ r_hat),                               # [N]
                  "rand": scale(post_u[:, -1, R_LAYER] @ r_hat)}                          # [N]

In [ ]:
print(f"base r (no push): {base.mean():.2f}    (0 = Eng harmless, 1 = Eng harmful)")
for mult, g_ in gate.items():
    print(f"{mult}x  Δ={mult*gap:6.1f} | push h: {g_['h'].mean():.2f} (frac>0.5: {(g_['h'] > 0.5).float().mean():.2f})"
          f" | push random: {g_['rand'].mean():.2f}")

In [ ]:
own_h = {}
for lang in langs:
    X = A[lang]
    v = (X["harmful_train"]["inst"][:, H_LAYER].mean(0)
         - X["harmless_train"]["inst"][:, H_LAYER].mean(0))     # [N, d] -> mean(0) -> [d]
    own_h[lang] = v / v.norm()                                   # [d], unit length
    print(f"{lang:9s} cos(own h, English h) = {(own_h[lang] @ h_hat).item():.3f}")

In [ ]:
res2 = {}   # res2[(lang, st)] -> {"base", "eng_h", "own_h", "random"}: [N] scaled r at L18
for lang in langs:
    pushes = {"eng_h": h_hat, "own_h": own_h[lang], "random": u_hat}   # each [d], unit length
    for st in sets:
        ps = [find_positions(tok, p, kw) for p in prompts[lang][st]]
        x_inst = A[lang][st]["inst"][:, H_LAYER]                        # [N, d], no-push, t_inst
        out = {"base": scale(A[lang][st]["last"][:, R_LAYER] @ r_hat)}  # [N]
        for key, v in pushes.items():
            inst, post = collect_resid_batched(bridge, ps, pad_id, hooks=span_hooks(v, gap))
            assert torch.allclose(inst[:, :H_LAYER], A[lang][st]["inst"][:, :H_LAYER], atol=1e-3)  # layers < 12 untouched
            assert torch.allclose(inst[:, H_LAYER] @ v, x_inst @ v + gap, atol=1e-2)               # push landed along v
            out[key] = scale(post[:, -1, R_LAYER] @ r_hat)              # [N]
        res2[(lang, st)] = out

In [ ]:
print(f"{'lang':9s} {'set':13s} {'base':>5s} {'eng_h':>6s} {'own_h':>6s} {'random':>7s} {'eng>0.7':>8s} {'own>0.7':>8s}")
for (lang, st), o in res2.items():
    print(f"{lang:9s} {st:13s} {o['base'].mean():5.2f} {o['eng_h'].mean():6.2f} {o['own_h'].mean():6.2f} "
          f"{o['random'].mean():7.2f} {(o['eng_h'] > 0.7).float().mean():8.2f} {(o['own_h'] > 0.7).float().mean():8.2f}")

In [ ]:
n_layers = bridge.cfg.n_layers                                   # 28 for Qwen3-1.7B, read from the model
R_READ = n_layers - 2                                            # late readout layer, after every push layer
step = n_layers // 7
PUSH_LAYERS = list(range(2 * step, R_READ, step))                # [8, 12, 16, 20, 24] here; scales with depth

H_all = dirs["h"] / dirs["h"].norm(dim=-1, keepdim=True)         # [n_layers, d] / [n_layers, 1] -> unit row per layer
r_read = dirs["r"][-1, R_READ] / dirs["r"][-1, R_READ].norm()    # [d], refusal direction at the readout layer
r_lo2 = (E["harmless_train"]["last"][:, R_READ] @ r_read).mean() # English harmless level at R_READ
r_hi2 = (E["harmful_train"]["last"][:, R_READ] @ r_read).mean()  # English harmful level at R_READ

def scale2(x):
    return (x - r_lo2) / (r_hi2 - r_lo2)                         # 0 = Eng harmless, 1 = Eng harmful (at R_READ)

gaps = {}
for L in PUSH_LAYERS:
    a = E["harmful_train"]["inst"][:, L] @ H_all[L]              # [N_harmful]
    b = E["harmless_train"]["inst"][:, L] @ H_all[L]             # [N_harmless]
    gaps[L] = (a.mean() - b.mean()).item()
    norm = E["harmless_train"]["inst"][:, L].norm(dim=-1).mean().item()
    print(f"L{L:2d}  Δ = {gaps[L]:6.1f}   residual norm {norm:7.1f}   Δ/norm {gaps[L] / norm:.2f}")
print(f"readout: r at L{R_READ}")

In [ ]:
def span_hooks_at(v, delta, layer):
    """Like span_hooks, but at any layer: add delta * v_hat to instruction tokens at blocks.{layer}.hook_resid_post."""
    def build(ps_, shift):
        start = torch.tensor([p["inst_start"] for p in ps_]) + shift    # [b]
        end   = torch.tensor([p["t_inst"]     for p in ps_]) + shift    # [b]
        return [(f"blocks.{layer}.hook_resid_post", make_span_add(v, delta, start, end))]
    return build

In [ ]:
sweep, base2 = {}, {}   # sweep[(lang, st, L)] -> {"h": [N], "rand": [N]};  base2[(lang, st)] -> [N]
for lang in langs:
    for st in sets:
        ps = [find_positions(tok, p, kw) for p in prompts[lang][st]]
        acts = A[lang][st]["inst"]                                      # [N, n_layers, d], no push, t_inst
        base2[(lang, st)] = scale2(A[lang][st]["last"][:, R_READ] @ r_read)   # [N]
        for L in PUSH_LAYERS:
            v, d = H_all[L], gaps[L]
            inst, post = collect_resid_batched(bridge, ps, pad_id, hooks=span_hooks_at(v, d, L))
            assert torch.allclose(inst[:, :L], acts[:, :L], atol=1e-3)                  # layers below L untouched
            assert torch.allclose(inst[:, L] @ v, acts[:, L] @ v + d, atol=1e-1)        # push landed at L
            _, post_u = collect_resid_batched(bridge, ps, pad_id, hooks=span_hooks_at(u_hat, d, L))
            sweep[(lang, st, L)] = {"h":    scale2(post[:, -1, R_READ] @ r_read),     # [N]
                                    "rand": scale2(post_u[:, -1, R_READ] @ r_read)}   # [N]
    print(lang, "done")

In [ ]:
print(f"r at L{R_READ}, English scale. Each cell: push h (random)")
print(f"{'lang':9s} {'set':13s} {'base':>5s} " + " ".join(f"{'L' + str(L):>12s}" for L in PUSH_LAYERS))
for lang in langs:
    for st in sets:
        cells = " ".join(f"{sweep[(lang, st, L)]['h'].mean():5.2f} ({sweep[(lang, st, L)]['rand'].mean():4.2f})"
                         for L in PUSH_LAYERS)
        print(f"{lang:9s} {st:13s} {base2[(lang, st)].mean():5.2f} {cells}")